# Gold — dimensões
Monta `dim_cliente`, `dim_vendedor`, `dim_produto` e `dim_unidade` a partir da Silver (somente registros não excluídos) e publica as descrições (COMMENT) no Unity Catalog.

In [ ]:
for nome, padrao in [("catalogo_silver", "dev_silver"), ("catalogo_gold", "dev_gold"), ("src_path", "")]:
    dbutils.widgets.text(nome, padrao)

import sys

src_path = dbutils.widgets.get("src_path")
if src_path and src_path not in sys.path:
    sys.path.append(src_path)

In [ ]:
from pyspark.sql import functions as F

from pdc_lib.comentarios import comandos_comentario
from pdc_lib.transformacoes import dim_unidade
from pdc_lib.util import nome_tabela, validar_identificador

catalogo_silver = validar_identificador(dbutils.widgets.get("catalogo_silver"))
catalogo_gold = validar_identificador(dbutils.widgets.get("catalogo_gold"))

origens = {t: nome_tabela(catalogo_silver, "protheus", t) for t in ["sa1", "sa3", "sb1"]}
if not all(spark.catalog.tableExists(t) for t in origens.values()):
    dbutils.notebook.exit("Silver dos cadastros ainda não disponível.")


def ativos(tabela: str):
    return spark.table(origens[tabela]).filter(~F.col("flg_deletado"))


def publicar(df, tabela: str) -> None:
    destino = nome_tabela(catalogo_gold, "comercial", tabela)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    for comando in comandos_comentario(destino, tabela):
        spark.sql(comando)
    print(f"{destino}: {df.count()} registros.")

In [ ]:
publicar(
    ativos("sa1").select("cod_empresa", "cod_cliente", "cod_loja", "des_nome", "des_nome_reduzido", "sig_uf",
                         "des_municipio", "cod_vendedor", "flg_bloqueado"),
    "dim_cliente",
)
publicar(ativos("sa3").select("cod_empresa", "cod_vendedor", "des_nome"), "dim_vendedor")
publicar(
    ativos("sb1").select("cod_empresa", "cod_produto", "des_produto", "des_tipo", "des_unidade_medida",
                         "cod_grupo", "vlr_preco_venda"),
    "dim_produto",
)
publicar(dim_unidade(spark), "dim_unidade")